> 📄 English: [intellagent-evaluation-tutorial.ipynb](intellagent-evaluation-tutorial.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial)

# 使用 IntellAgent 评估 LLM Agent：完整教程

## 概述

本 notebook 将引导你使用一个强大的多 agent 评估框架来构建和评估一个 AI agent。

## 什么是 IntellAgent？

[IntellAgent](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=plurai&target=https%3A%2F%2Fintellagent-doc.plurai.ai%2F&text=IntellAgent) 是一个创新的多 agent 框架，可在模拟环境中自动评估 AI agent。可以把它看作一个全面且先进的测试套件，能够生成真实场景、模拟用户交互，并对你的 agent 表现进行深入分析。

## 挑战：为什么 Agent 评估如此困难

评估 LLM agent 面临两大挑战：
1. **数据集构建**：创建多样化、有挑战性且真实的测试场景，覆盖各种边界情况
2. **可靠指标**：定义能够准确衡量 agent 表现（而不仅仅是简单准确率）的评估标准

传统的测试方法往往力不从心，因为它们依赖静态数据集，无法捕捉真实用户交互的动态特性。

## IntellAgent 的三阶段方法
IntellAgent 通过一个复杂的三阶段流程来解决这些挑战：
![arch_overview.png](attachment:ff8fdf84-fa85-4c39-88fd-c814c7382884.png)

### 🎯 阶段 1：智能场景生成
- 自动创建真实、多样化的测试场景
- 生成测试特定规则和策略的边界情况
- 确保对潜在用户交互的全面覆盖

### 🤖 阶段 2：动态模拟
- 模拟真实的用户与 agent 对话
- 根据 agent 的响应调整交互模式
- 捕捉复杂的多轮对话动态

### 📊 阶段 3：细粒度分析
- 提供详细的行为分析
- 识别策略违规和性能差距
- 提供可操作的改进建议

## 主要优势

- **自动化测试**：无需手动创建测试用例
- **全面覆盖**：测试你可能想不到的场景
- **真实评估**：模拟真实的用户行为模式
- **可操作洞察**：清晰地反馈需要改进的地方
- **可扩展**：从简单聊天机器人到复杂的多工具 agent 都能适用

## 你将学到什么

在本教程中，我们将构建并评估一个简单的教育助手 agent。你将学会如何：

1. 搭建 IntellAgent 框架
2. 配置模拟参数
3. 运行自动化评估
4. 使用内置的可视化工具分析结果

让我们开始，看看 IntellAgent 的实际表现吧！

📖 **想了解更多关于 IntellAgent 多 agent 框架和评估的背景知识，请查看我们的详细博客文章：** [IntellAgent multi-agent framework](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=substack-open&target=https%3A%2F%2Fopen.substack.com%2Fpub%2Fdiamantai%2Fp%2Fintellagent-the-multi-agent-framework%3Fr%3D336pe4%26utm_campaign%3Dpost%26utm_medium%3Dweb%26showWelcomeOnShare%3Dfalse&text=IntellAgent%20multi-agent%20framework)

*DiamantAI 是一份面向 AI 领域的顶级（前 0.1%）通讯，拥有超过 25,000 名订阅者，专注于 AI 技术、突破和教程。*

---

## 步骤 1：克隆 IntellAgent 仓库

首先，我们需要在环境中搭建 IntellAgent 框架。这涉及克隆仓库并安装所有必要的依赖。

In [ ]:
# Clone the repository
!git clone https://github.com/plurai-ai/intellagent.git
# Change to the repo directory
%cd intellagent

In [ ]:
# Install the required dependencies
!pip install -r requirements.txt && pip install nest_asyncio

## 步骤 2：配置 LLM API 凭据

IntellAgent 需要访问大语言模型（LLM）来生成场景、模拟用户并运行你的 agent。该框架支持所有主流 LLM 提供商，包括 OpenAI、Anthropic、Azure、Google、Bedrock 和 NVIDIA。

我们将创建一个配置文件来安全地存储你的 API 凭据。请务必将占位符替换为你实际的 API key。

In [ ]:
import os
import yaml

# # Enter your OpenAI API key (the framework support all Langchain LLM providers)
OPENAI_API_KEY = "your-api-key-here"  # Replace with your actual API key

#Create the llm_env.yml configuration file
llm_config = {
    "openai": {
        "OPENAI_API_KEY": OPENAI_API_KEY
    }
}


# Make sure the config directory exists
os.makedirs("config", exist_ok=True)

# Write the configuration to the file
with open("config/llm_env.yml", "w") as f:
    yaml.dump(llm_config, f)

print("✅ LLM API credentials configured successfully.")
print("📝 Note: You can add multiple providers to this configuration file.")

## 步骤 3：设计你的 Agent

接下来是最令人兴奋的部分 —— 定义你的 agent 应该做什么！我们将创建一个帮助学生学习的学习助手。下面的提示定义了 agent 的个性、能力，以及最重要的、它应遵循的策略。

**为什么这很重要**：提示至关重要，因为 IntellAgent 会测试你的 agent 是否真的遵循这些准则。它会生成试图让 agent 违反其策略的场景，帮助你在部署前识别潜在问题。

请注意我们如何定义了「不要直接解题」和「使用适龄语言」等具体策略 —— IntellAgent 将创建测试场景来验证这些行为。

In [ ]:
# Create the necessary directories
os.makedirs("examples/my_education_agent/input", exist_ok=True)

# Define a simple educational chatbot prompt
education_prompt = """
# Educational Assistant Guidelines

You are an educational assistant designed to help students with their learning needs. Follow these guidelines:

## Core Responsibilities:
- Provide clear, accurate information on educational topics
- Explain complex concepts in simple terms
- Help with homework questions by guiding the student through the solution process
- Recommend learning resources when appropriate

## Policies:
1. **Do not solve problems directly** - Instead, provide guidance and hints
2. **Use age-appropriate language** - Adjust explanations based on the student's level
3. **Encourage critical thinking** - Ask follow-up questions that promote deeper understanding
4. **Be patient and supportive** - Create a positive learning environment
5. **Verify understanding** - Check if the student has understood the explanation

## Subject Areas:
- Mathematics (Basic arithmetic to advanced calculus)
- Science (Physics, Chemistry, Biology)
- Language Arts (Grammar, Writing, Literature)
- Social Studies (History, Geography, Civics)

Remember, your goal is to help students learn and grow, not just provide answers.
"""

# Write the prompt to a file
with open("examples/my_education_agent/input/wiki.md", "w") as f:
    f.write(education_prompt)

print("✅ Educational agent prompt created successfully.")
print("🎯 Agent will be tested on policies like 'no direct problem solving' and 'age-appropriate language'")

## 步骤 4：配置模拟参数

在运行评估之前，我们需要告诉 IntellAgent 如何进行测试。这个配置文件控制着几个重要方面：

- **使用哪些模型**：我们可以为 IntellAgent 系统和你的实际 agent 使用不同的模型
- **多少个测试场景**：场景越多 = 测试越全面，但运行时间也更长

我们从一个保守的设置开始，以保证教程能快速运行，但你可以在生产测试时扩大这些设置。

In [ ]:
# Define the configuration
config = {
    "environment": {
        "prompt_path": "examples/my_education_agent/input/wiki.md",  # Path to our agent's prompt
    },
    "llm_intellagent": {
        "type": "openai",  # Using OpenAI for the IntellAgent system
        "name": "gpt-4o"   # You can adjust this to another model if needed
    },
    "llm_chat": {
        "type": "openai",  # Using OpenAI for the chatbot agent
        "name": "gpt-4o-mini"   # You can adjust this to another model if needed
    },
    "dataset": {
        "num_samples": 10   # Number of test scenarios to generate (reduced for faster execution)
    }
}

# Write the configuration to a file
with open("config/my_education_config.yml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Configuration file created successfully.")
print(f"🔧 Will generate {config['dataset']['num_samples']} test scenarios")

## 步骤 5：初始化 IntellAgent 系统

现在我们准备好开始评估流程了！这一步使用我们的配置来初始化 IntellAgent 框架。系统将准备根据你的 agent 提示和策略来生成测试场景。

**这里发生了什么**：IntellAgent 分析你的 agent 提示，以了解它应该测试哪些策略和行为。然后它准备创建多样化的场景，来挑战你 agent 的这些方面。

In [ ]:
import nest_asyncio
import warnings
warnings.filterwarnings(
    "ignore",
    message="API key must be provided when using hosted LangSmith API"
)
nest_asyncio.apply()

### 加载模拟环境

这个单元格加载我们的配置并初始化模拟环境。`SimulatorExecutor` 是协调整个评估流程的主引擎。

In [ ]:
from simulator.utils.file_reading import override_config
from simulator.simulator_executor import SimulatorExecutor
import importlib
import simulator.utils.logger_config as logger_config
importlib.reload(logger_config)
base_output_path = './results/education'
config = override_config('config/my_education_config.yml') # Loading the config file
# loading the simulator executor with the environment
executor = SimulatorExecutor(config, base_output_path) 

print("✅ IntellAgent simulation environment initialized")
print(f"📁 Results will be saved to: {base_output_path}")

## 步骤 6：生成合成数据集
我们生成一个场景数据库，并打印第一个场景

In [ ]:
# This will generate the dataset in case the dataset was not generated before
executor.load_dataset('data_1')

print("✅ Finish generating the scenario dataset")
print(f"🧐 An example of generated scenario: \n{executor.dataset_handler.records[0].description.event_description}")

## 步骤 7：模拟与 Agent 的交互

接下来，我们模拟交互，并应用一个评审来评估是否有任何策略被违反。

In [ ]:
# Run the complete simulation process
print("🚀 Starting simulation...")
print("📊 This will generate scenarios, simulate conversations, and analyze results")
print("⏱️  Estimated time: 2-5 minutes for 10 scenarios")
print("\n" + "="*50)

executor.run_simulation('exp_1')

print("\n" + "="*50)
print("✅ Simulation completed successfully!")
print("📈 Results are ready for analysis")

## 步骤 8：分析你的结果

恭喜！你的 agent 已经经过了全面测试。现在是时候看看它的表现了。IntellAgent 提供了一个全面的可视化仪表盘，让你可以探索评估的各个方面。

**你将在仪表盘中看到的内容**：
- 展示具体用户与 agent 交互的单个对话轨迹
- 策略违规分析，突出显示你的 agent 未遵循准则的地方
- 跨不同类型策略和场景的性能指标
- 详细评审，解释哪些地方做得好、哪些地方需要改进

可视化将在你的浏览器中启动，并提供一个交互式界面来探索所有结果。

In [ ]:
# Launch the results visualization dashboard
import subprocess
import threading
import time
from IPython.display import IFrame

def run_streamlit():
    try:
        subprocess.run([
            "streamlit", "run", "simulator/visualization/Simulator_Visualizer.py"
        ], cwd=".")
    except Exception as e:
        print(f"Error running Streamlit: {e}")

# Start Streamlit in a separate thread
print("🚀 Starting IntellAgent Results Dashboard...")
print("📊 This will launch an interactive visualization of your agent's performance")
print("⏱️  Please wait a moment for the dashboard to load ")

streamlit_thread = threading.Thread(target=run_streamlit)
streamlit_thread.daemon = True  # Set as daemon so it exits when the notebook exits
streamlit_thread.start()

# Wait a few seconds for Streamlit to start
time.sleep(5)

# Display in an iframe (may not work in all environments)
try:
    display(IFrame(src="http://localhost:8501", width=1000, height=600))
    print("\n📈 Dashboard loaded successfully!")
    print("🔍 Navigate to the '🤖 Session Visualizer' page to explore conversation traces")
except:
    print("\n🌐 Dashboard is running at: http://localhost:8501")
    print("📝 Please open this URL in a new browser tab to view your results")
    print("🔍 Once there, navigate to the '🤖 Session Visualizer' page")

## 结语与下一步

🎉 **恭喜！** 你已经成功完成了你的第一次 IntellAgent 评估。你现在拥有了：

✅ 对 IntellAgent 工作原理的全面理解
✅ 完整评估流程的动手实践
✅ 对 agent 表现的详细分析
✅ 可操作的改进建议

### 下一步

为了进一步提升你的评估流程，你可以：

- 尝试不同的 agent 提示或指令
- 尝试不同的 LLM 模型（例如 GPT-4o、Claude 等）
- 测试一个与数据库和工具交互的更复杂 agent（参见 [airline 示例](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=plurai-airline&target=https%3A%2F%2Fintellagent-doc.plurai.ai%2Fexamples%2Fairline%2F&text=airline%20example)）
- [自定义](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=plurai-custom-chatbot&target=https%3A%2F%2Fintellagent-doc.plurai.ai%2Fcustomization%2Fcustom_chatbot%2F&text=Customize)该框架以适配你的具体用例


### 获取高级支持

需要一个最先进的、定制化的评估框架，用真实数据测试 agent 的所有组件，并提供量身定制的评审优化？

我们的服务包括：
- **复杂多工具 Agent**：测试使用数据库、API 和外部工具的 agent
- **RAG 系统评估**：评估检索增强生成（retrieval-augmented generation）的性能
- **自定义评估指标**：定义特定领域的成功标准
- **真实数据集成**：用你实际的用户交互模式进行测试


---

**资源**：[IntellAgent GitHub 仓库](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=github-intellagent&target=https%3A%2F%2Fgithub.com%2Fplurai-ai%2Fintellagent&text=IntellAgent%20GitHub%20Repository) | [联系 Plurai](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-evaluation-intellagent--intellagent-evaluation-tutorial&click=plurai-contact-us&target=https%3A%2F%2Fwww.plurai.ai%2Fcontact-us&text=Contact%20Plurai)